In [37]:
#import sys
#!{sys.executable} -m pip install soccerdata

In [38]:
import soccerdata as sd
leagues = sd.FBref.available_leagues()

In [39]:
import soccerdata as sd
print(sd.FBref.available_leagues())

['Big 5 European Leagues Combined', 'ENG-Premier League', 'ESP-La Liga', 'FRA-Ligue 1', 'GER-Bundesliga', 'INT-European Championship', "INT-Women's World Cup", 'INT-World Cup', 'ITA-Serie A']


In [40]:
import soccerdata as sd
import pandas as pd

# 1. Configuration : Récupération des données FBref
fb = sd.FBref(leagues=["Big 5 European Leagues Combined"], seasons="2025-26")

# Récupération des différents blocs de stats
shooting = fb.read_player_season_stats(stat_type="shooting")
standard = fb.read_player_season_stats(stat_type="standard")
playing = fb.read_player_season_stats(stat_type="playing_time")

# 2. Fusion étape par étape (pour éviter l'erreur de suffixe)
df = standard.join(shooting, how='inner', rsuffix='_shoot')
df = df.join(playing, how='inner', rsuffix='_play')

# 3. Aplatissement des colonnes (Indispensable pour simplifier les calculs)
# Transforme ('Performance', 'Gls') en 'Performance_Gls'
df.columns = ['_'.join(col).strip() for col in df.columns.values]

# 4. Nettoyage et création des métriques par 90 minutes
# On s'assure d'avoir assez de temps de jeu (ex: > 450 min) pour ne pas fausser les stats
df = df[df['Playing Time_Min'] > 450].copy()

# Calculs des métriques clés
# Note : nous utilisons les colonnes déjà calculées par FBref quand elles existent ('Per 90 Minutes')
df['goals_p90'] = df['Per 90 Minutes_Gls']
df['assists_p90'] = df['Per 90 Minutes_Ast']
df['selection'] = df['nation_play_']
df['goal_assist_p90'] = df['Per 90 Minutes_G+A']
df['team_success'] = df['Team Success_onGA']




# Ajout des penaltys
df['pen_scored'] = df['Performance_PK']
df['pen_attempted'] = df['Performance_PKatt']

# 5. Sélection finale pour le modèle
# On choisit les colonnes les plus prédictives pour un modèle de performance
features = [
    'goals_p90', 'assists_p90', 'selection', 'goal_assist_p90', 'team_success'
]

model_df = df[features].fillna(0) # On remplace les données manquantes par 0

print("--- Données prêtes pour le modèle ---")
print(model_df.head())

# 6. Sauvegarde
model_df.to_csv("stats_joueurs_modele_final2.csv")
print("\nFichier sauvegardé avec succès dans 'data/stats_joueurs_modele_final2.csv'")

[06/05/26 08:59:48] INFO     Saving cached data to C:\Users\user\soccerdata\data\FBref               ]8;id=6579767;file://c:\Users\user\Projet3\.venv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=6579768;file://c:\Users\user\Projet3\.venv\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

--- Données prêtes pour le modèle ---
                                                      goals_p90  assists_p90  \
league             season team    player                                       
ENG-Premier League 2526   Arsenal Ben White                 0.0         0.13   
                                  Bukayo Saka              0.28          0.2   
                                  Cristhian Mosquera        0.0          0.0   
                                  David Raya                0.0          0.0   
                                  Declan Rice              0.12         0.15   

                                                     selection  \
league             season team    player                         
ENG-Premier League 2526   Arsenal Ben White                ENG   
                                  Bukayo Saka              ENG   
                                  Cristhian Mosquera       ESP   
                                  David Raya               ESP   
     

In [41]:
print(df.columns.tolist())

['nation_', 'pos_', 'age_', 'born_', 'Playing Time_MP', 'Playing Time_Starts', 'Playing Time_Min', 'Playing Time_90s', 'Performance_Gls', 'Performance_Ast', 'Performance_G+A', 'Performance_G-PK', 'Performance_PK', 'Performance_PKatt', 'Performance_CrdY', 'Performance_CrdR', 'Per 90 Minutes_Gls', 'Per 90 Minutes_Ast', 'Per 90 Minutes_G+A', 'Per 90 Minutes_G-PK', 'Per 90 Minutes_G+A-PK', 'nation_shoot_', 'pos_shoot_', 'age_shoot_', 'born_shoot_', '90s_', 'Standard_Gls', 'Standard_Sh', 'Standard_SoT', 'Standard_SoT%', 'Standard_Sh/90', 'Standard_SoT/90', 'Standard_G/Sh', 'Standard_G/SoT', 'Standard_PK', 'Standard_PKatt', 'nation_play_', 'pos_play_', 'age_play_', 'born_play_', 'Playing Time_play_MP', 'Playing Time_play_Min', 'Playing Time_play_Mn/MP', 'Playing Time_play_Min%', 'Playing Time_play_90s', 'Starts_Starts', 'Starts_Mn/Start', 'Starts_Compl', 'Subs_Subs', 'Subs_Mn/Sub', 'Subs_unSub', 'Team Success_PPM', 'Team Success_onG', 'Team Success_onGA', 'Team Success_+/-', 'Team Success_+/

In [42]:
df.columns

Index(['nation_', 'pos_', 'age_', 'born_', 'Playing Time_MP',
       'Playing Time_Starts', 'Playing Time_Min', 'Playing Time_90s',
       'Performance_Gls', 'Performance_Ast', 'Performance_G+A',
       'Performance_G-PK', 'Performance_PK', 'Performance_PKatt',
       'Performance_CrdY', 'Performance_CrdR', 'Per 90 Minutes_Gls',
       'Per 90 Minutes_Ast', 'Per 90 Minutes_G+A', 'Per 90 Minutes_G-PK',
       'Per 90 Minutes_G+A-PK', 'nation_shoot_', 'pos_shoot_', 'age_shoot_',
       'born_shoot_', '90s_', 'Standard_Gls', 'Standard_Sh', 'Standard_SoT',
       'Standard_SoT%', 'Standard_Sh/90', 'Standard_SoT/90', 'Standard_G/Sh',
       'Standard_G/SoT', 'Standard_PK', 'Standard_PKatt', 'nation_play_',
       'pos_play_', 'age_play_', 'born_play_', 'Playing Time_play_MP',
       'Playing Time_play_Min', 'Playing Time_play_Mn/MP',
       'Playing Time_play_Min%', 'Playing Time_play_90s', 'Starts_Starts',
       'Starts_Mn/Start', 'Starts_Compl', 'Subs_Subs', 'Subs_Mn/Sub',
       'Subs_u

In [43]:
# Permet d'afficher jusqu'à 100 colonnes (ou plus)
pd.set_option('display.max_columns', 100) 
# Permet d'afficher le contenu complet des colonnes (évite les ...)
pd.set_option('display.max_colwidth', None)

# Ou affichez le début du dataframe
display(df.sample(20))

nation_  \
league             season team                player                         
ESP-La Liga        2526   Espanyol            Jofre                    ESP   
ITA-Serie A        2526   Napoli              Rasmus Højlund           DEN   
nan                2526   Eintracht Frankfurt Robin Koch               GER   
ESP-La Liga        2526   Oviedo              David Carmo              ANG   
ENG-Premier League 2526   Burnley             Bashir Humphreys         ENG   
ITA-Serie A        2526   Pisa                İsak Vural               TUR   
ESP-La Liga        2526   Girona              Arnau Martinez           ESP   
                          Athletic Club       Aymeric Laporte          ESP   
nan                2526   Werder Bremen       Isaac Schmidt            SUI   
ITA-Serie A        2526   Torino              Marcus Pedersen          NOR   
                          Bologna             Thijs Dallinga           NED   
ENG-Premier League 2526   Crystal Palace      Yeremi Pino              ESP   
FRA-Ligue 1        2526   Lens                Jonathan Gradit          FRA   
ENG-Premier League 2526   Nottingham Forest   Jair Cunha               BRA   
FRA-Ligue 1        2526   Nice                Hicham Boudaoui          ALG   
ITA-Serie A        2526   Bologna             Lorenzo De Silvestri     ITA   
nan                2526   Stuttgart           Finn Jeltsch             GER   
ENG-Premier League 2526   Nottingham Forest   Dilane Bakwa             FRA   
ESP-La Liga        2526   Getafe              Mauro Arambarri          URU   
ENG-Premier League 2526   Liverpool           Andrew Robertson         SCO   

                                                                     pos_  \
league             season team                player                        
ESP-La Liga        2526   Espanyol            Jofre                    MF   
ITA-Serie A        2526   Napoli              Rasmus Højlund           FW   
nan                2526   Eintracht Frankfurt Robin Koch               DF   
ESP-La Liga        2526   Oviedo              David Carmo              DF   
ENG-Premier League 2526   Burnley             Bashir Humphreys      DF,MF   
ITA-Serie A        2526   Pisa                İsak Vural               MF   
ESP-La Liga        2526   Girona              Arnau Martinez        DF,MF   
                          Athletic Club       Aymeric Laporte          DF   
nan                2526   Werder Bremen       Isaac Schmidt         MF,DF   
ITA-Serie A        2526   Torino              Marcus Pedersen       MF,DF   
                          Bologna             Thijs Dallinga           FW   
ENG-Premier League 2526   Crystal Palace      Yeremi Pino           MF,FW   
FRA-Ligue 1        2526   Lens                Jonathan Gradit          DF   
ENG-Premier League 2526   Nottingham Forest   Jair Cunha               DF   
FRA-Ligue 1        2526   Nice                Hicham Boudaoui          MF   
ITA-Serie A        2526   Bologna             Lorenzo De Silvestri     DF   
nan                2526   Stuttgart           Finn Jeltsch             DF   
ENG-Premier League 2526   Nottingham Forest   Dilane Bakwa             MF   
ESP-La Liga        2526   Getafe              Mauro Arambarri          MF   
ENG-Premier League 2526   Liverpool           Andrew Robertson         DF   

                                                                    age_  \
league             season team                player                       
ESP-La Liga        2526   Espanyol            Jofre                   24   
ITA-Serie A        2526   Napoli              Rasmus Højlund          22   
nan                2526   Eintracht Frankfurt Robin Koch              29   
ESP-La Liga        2526   Oviedo              David Carmo             26   
ENG-Premier League 2526   Burnley             Bashir Humphreys        22   
ITA-Serie A        2526   Pisa                İsak Vural              19   
ESP-La Liga        2526   Girona              A

In [ ]:
# Avant : vos index sont 'league', 'season', 'team', 'player'
# Après : ils deviennent des colonnes standards
df = df.reset_index()

# Vous pouvez maintenant manipuler 'season' comme n'importe quelle colonne
df.columns

,index,league,season,team,player,nation_,pos_,age_,born_,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,Performance_G+A,Performance_G-PK,Performance_PK,Performance_PKatt,Performance_CrdY,Performance_CrdR,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,nation_shoot_,pos_shoot_,age_shoot_,born_shoot_,90s_,Standard_Gls,Standard_Sh,Standard_SoT,Standard_SoT%,Standard_Sh/90,Standard_SoT/90,Standard_G/Sh,Standard_G/SoT,Standard_PK,Standard_PKatt,nation_play_,pos_play_,age_play_,born_play_,Playing Time_play_MP,Playing Time_play_Min,Playing Time_play_Mn/MP,Playing Time_play_Min%,Playing Time_play_90s,Starts_Starts,Starts_Mn/Start,Starts_Compl,Subs_Subs,Subs_Mn/Sub,Subs_unSub,Team Success_PPM,Team Success_onG,Team Success_onGA,Team Success_+/-,Team Success_+/-90,Team Success_On-Off,goals_p90,assists_p90,selection,goal_assist_p90,team_success,pen_scored,pen_attempted
0,0,ENG-Premier League,2526,Arsenal,Ben White,ENG,DF,27,1997,12,9,702,7.8,0,1,1,0,0,0,0,0,0.0,0.13,0.13,0.0,0.13,ENG,DF,27,1997,7.8,0,5,1,20.0,0.64,0.13,0.0,0.0,0,0,ENG,DF,27,1997,12,702,59,20.5,7.8,9,72,4,3,19,17,1.83,10,6,4,0.51,-0.81,0.0,0.13,ENG,0.13,6,0,0
1,1,ENG-Premier League,2526,Arsenal,Bukayo Saka,ENG,"FW,MF",23,2001,31,25,2222,24.7,7,5,12,6,1,1,2,0,0.28,0.2,0.49,0.24,0.45,ENG,"FW,MF",23,2001,24.7,7,71,27,38.0,2.88,1.09,0.08,0.22,1,1,ENG,"FW,MF",23,2001,31,2222,72,65.0,24.7,25,82,15,6,27,1,2.35,46,16,30,1.22,0.16,0.28,0.2,ENG,0.49,16,1,1
2,2,ENG-Premier League,2526,Arsenal,Cristhian Mosquera,ESP,DF,21,2004,20,9,994,11.0,0,0,0,0,0,0,4,0,0.0,0.0,0.0,0.0,0.0,ESP,DF,21,2004,11.0,0,1,0,0.0,0.09,0.0,0.0,<NA>,0,0,ESP,DF,21,2004,20,994,50,29.1,11.0,9,75,5,11,29,11,2.15,17,8,9,0.81,-0.48,0.0,0.0,ESP,0.0,8,0,0
3,3,ENG-Premier League,2526,Arsenal,David Raya,ESP,GK,29,1995,37,37,3330,37.0,0,0,0,0,0,0,1,0,0.0,0.0,0.0,0.0,0.0,ESP,GK,29,1995,37.0,0,0,0,<NA>,0.0,0.0,<NA>,<NA>,0,0,ESP,GK,29,1995,37,3330,90,97.4,37.0,37,90,37,0,<NA>,0,2.22,69,26,43,1.16,0.16,0.0,0.0,ESP,0.0,26,0,0
4,4,ENG-Premier League,2526,Arsenal,Declan Rice,ENG,MF,26,1999,36,35,3094,34.4,4,5,9,4,0,0,3,0,0.12,0.15,0.26,0.12,0.26,ENG,MF,26,1999,34.4,4,40,13,32.5,1.16,0.38,0.1,0.31,0,0,ENG,MF,26,1999,36,3094,86,90.5,34.4,35,88,29,1,23,1,2.19,62,25,37,1.08,-0.86,0.12,0.15,ENG,0.26,25,0,0


In [52]:
df.drop(columns=['nation_', 'Performance_PK', 'Performance_PKatt', 'Standard_PK', 'Standard_PKatt', 'Subs_Subs', 'Subs_Mn/Sub', 'Subs_unSub', 'index', 'nation_shoot_', 'pos_shoot_', 'age_shoot_', 'born_shoot_'])

,league,season,team,player,pos_,age_,born_,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,Performance_G+A,Performance_G-PK,Performance_CrdY,Performance_CrdR,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,90s_,Standard_Gls,Standard_Sh,Standard_SoT,Standard_SoT%,Standard_Sh/90,Standard_SoT/90,Standard_G/Sh,Standard_G/SoT,nation_play_,pos_play_,age_play_,born_play_,Playing Time_play_MP,Playing Time_play_Min,Playing Time_play_Mn/MP,Playing Time_play_Min%,Playing Time_play_90s,Starts_Starts,Starts_Mn/Start,Starts_Compl,Team Success_PPM,Team Success_onG,Team Success_onGA,Team Success_+/-,Team Success_+/-90,Team Success_On-Off,goals_p90,assists_p90,selection,goal_assist_p90,team_success,pen_scored,pen_attempted
0,ENG-Premier League,2526,Arsenal,Ben White,DF,27,1997,12,9,702,7.8,0,1,1,0,0,0,0.0,0.13,0.13,0.0,0.13,7.8,0,5,1,20.0,0.64,0.13,0.0,0.0,ENG,DF,27,1997,12,702,59,20.5,7.8,9,72,4,1.83,10,6,4,0.51,-0.81,0.0,0.13,ENG,0.13,6,0,0
1,ENG-Premier League,2526,Arsenal,Bukayo Saka,"FW,MF",23,2001,31,25,2222,24.7,7,5,12,6,2,0,0.28,0.2,0.49,0.24,0.45,24.7,7,71,27,38.0,2.88,1.09,0.08,0.22,ENG,"FW,MF",23,2001,31,2222,72,65.0,24.7,25,82,15,2.35,46,16,30,1.22,0.16,0.28,0.2,ENG,0.49,16,1,1
2,ENG-Premier League,2526,Arsenal,Cristhian Mosquera,DF,21,2004,20,9,994,11.0,0,0,0,0,4,0,0.0,0.0,0.0,0.0,0.0,11.0,0,1,0,0.0,0.09,0.0,0.0,<NA>,ESP,DF,21,2004,20,994,50,29.1,11.0,9,75,5,2.15,17,8,9,0.81,-0.48,0.0,0.0,ESP,0.0,8,0,0
3,ENG-Premier League,2526,Arsenal,David Raya,GK,29,1995,37,37,3330,37.0,0,0,0,0,1,0,0.0,0.0,0.0,0.0,0.0,37.0,0,0,0,<NA>,0.0,0.0,<NA>,<NA>,ESP,GK,29,1995,37,3330,90,97.4,37.0,37,90,37,2.22,69,26,43,1.16,0.16,0.0,0.0,ESP,0.0,26,0,0
4,ENG-Premier League,2526,Arsenal,Declan Rice,MF,26,1999,36,35,3094,34.4,4,5,9,4,3,0,0.12,0.15,0.26,0.12,0.26,34.4,4,40,13,32.5,1.16,0.38,0.1,0.31,ENG,MF,26,1999,36,3094,86,90.5,34.4,35,88,29,2.19,62,25,37,1.08,-0.86,0.12,0.15,ENG,0.26,25,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1988,NaN,2526,Wolfsburg,Moritz Jenz,DF,26,1999,22,20,1787,19.9,1,0,1,1,8,1,0.05,0.0,0.05,0.05,0.05,19.9,1,5,1,20.0,0.25,0.05,0.2,1.0,GER,DF,26,1999,22,1787,81,58.4,19.9,20,89,18,0.68,21,44,-23,-1.16,-1.09,0.05,0.0,GER,0.05,44,0,0
1989,NaN,2526,Wolfsburg,Patrick Wimmer,MF,24,2001,25,21,1621,18.0,4,3,7,4,4,0,0.22,0.17,0.39,0.22,0.39,18.0,4,24,8,33.3,1.33,0.44,0.17,0.5,AUT,MF,24,2001,25,1621,65,53.0,18.0,21,73,3,1.0,27,32,-5,-0.28,0.91,0.22,0.17,AUT,0.39,32,0,0
1990,NaN,2526,Wolfsburg,Sael Kumbedi,"DF,MF",20,2005,24,20,1804,20.0,0,5,5,0,2,0,0.0,0.25,0.25,0.0,0.25,20.0,0,5,0,0.0,0.25,0.0,0.0,<NA>,FRA,"DF,MF",20,2005,24,1804,75,59.0,20.0,20,86,12,0.88,30,45,-15,-0.75,-0.1,0.0,0.25,FRA,0.25,45,0,0
1991,NaN,2526,Wolfsburg,Vinicius Souza,MF,26,1999,24,22,1764,19.6,0,0,0,0,5,0,0.0,0.0,0.0,0.0,0.0,19.6,0,9,2,22.2,0.46,0.1,0.0,0.0,BRA,MF,26,1999,24,1764,74,57.6,19.6,22,80,12,0.88,24,35,-11,-0.56,0.34,0.0,0.0,BRA,0.0,35,0,0
